# Phase-2 Complete Pipeline

This notebook runs Steps 1-4 in sequence with all fixes applied.

**Steps:**
1. Data Audit & Verification
2. Multi-Dataset Harmonization  
3. Baseline Models (with proper within-battery split)
4. Multimodal Model

**Key Fixes Applied:**
- Proper within-battery time-split for model validation
- Cross-battery split for generalization testing (Step 6)
- Correct normalization using training stats only
- Traditional ML models that work better on small datasets

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP
# ============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Verify data exists
assert os.path.exists('NASA_DataSets'), "NASA_DataSets folder not found!"
assert os.path.exists('phase2_config.yaml'), "Config not found!"

with open('phase2_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("✅ Setup complete")

---
# Step 1: Data Audit
---

In [ ]:
from data_audit import DataAudit

print("="*70)
print("STEP 1: DATA AUDIT")
print("="*70)

# DataAudit only takes config - PROJECT_ROOT is used via os.chdir above
auditor = DataAudit(config)

# Audit all batteries
batteries = ['nasa_train', 'nasa_test_1', 'nasa_test_2']
audit_results = {}

for bat_key in batteries:
    bat_config = config['datasets'][bat_key]
    result = auditor.audit_nasa_battery(bat_config['path'], bat_config['battery_id'])
    audit_results[bat_config['battery_id']] = result
    print()

In [ ]:
# Check all passed
all_passed = all(r['passed'] for r in audit_results.values())

print("\n" + "="*70)
print("AUDIT SUMMARY")
print("="*70)

for bat_id, result in audit_results.items():
    status = "✅ PASSED" if result['passed'] else "❌ FAILED"
    warnings_count = len(result.get('warnings', []))
    print(f"  {bat_id}: {status} ({warnings_count} warnings)")

if all_passed:
    print("\n✅ ALL AUDITS PASSED - Proceeding to Step 2")
else:
    print("\n⚠️ Some audits have issues - check warnings above")

---
# Step 2: Data Harmonization
---

In [ ]:
from data_harmonization import DataHarmonizer

print("="*70)
print("STEP 2: DATA HARMONIZATION")
print("="*70)

harmonizer = DataHarmonizer(config, PROJECT_ROOT)

# Load all batteries
battery_configs = [
    config['datasets']['nasa_train'],
    config['datasets']['nasa_test_1'],
    config['datasets']['nasa_test_2']
]

batteries = harmonizer.load_and_harmonize_nasa(battery_configs)

In [ ]:
# Extract features
features_df = harmonizer.extract_features()
print(f"\nTotal samples: {len(features_df)}")
print(f"Features: {len(features_df.columns)}")

In [ ]:
# Save ALL features (not just train/test split)
# This allows us to create proper splits later
os.makedirs('data/phase2/harmonized', exist_ok=True)

# Save the combined features
features_df.to_csv('data/phase2/harmonized/all_features.csv', index=False)
print("✅ Saved: data/phase2/harmonized/all_features.csv")

# Also save per-battery for convenience
for bat_id in features_df['battery_id'].unique():
    bat_df = features_df[features_df['battery_id'] == bat_id]
    bat_df.to_csv(f'data/phase2/harmonized/{bat_id}_features.csv', index=False)
    print(f"✅ Saved: data/phase2/harmonized/{bat_id}_features.csv ({len(bat_df)} samples)")

In [ ]:
# Visualize SOH across batteries
fig, ax = plt.subplots(figsize=(12, 5))

colors = {'B0005': 'blue', 'B0006': 'orange', 'B0007': 'green'}

for bat_id in features_df['battery_id'].unique():
    bat_df = features_df[features_df['battery_id'] == bat_id]
    ax.plot(bat_df['cycle_number'], bat_df['soh'], 'o-', 
            label=f'{bat_id} (n={len(bat_df)})', color=colors.get(bat_id, 'gray'), 
            markersize=3, alpha=0.7)

ax.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='EOL (80%)')
ax.set_xlabel('Cycle Number')
ax.set_ylabel('SOH (%)')
ax.set_title('SOH Degradation: All Batteries', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase2/figures/step2_soh_all.png', dpi=150)
plt.show()

---
# Step 3: Baseline Models
---

**Important:** We use TWO evaluation modes:
1. **Within-Battery Split** - Validates model architecture (should get good R²)
2. **Cross-Battery Split** - Tests generalization (harder, lower R² expected)

In [ ]:
from data_splits import create_within_battery_split, normalize_features
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("="*70)
print("STEP 3: BASELINE MODELS")
print("="*70)

In [ ]:
# Load all features
all_features = pd.read_csv('data/phase2/harmonized/all_features.csv')
print(f"Loaded {len(all_features)} samples from {all_features['battery_id'].nunique()} batteries")

# Create WITHIN-BATTERY split (for model validation)
splits = create_within_battery_split(all_features, train_ratio=0.7, val_ratio=0.15)

In [ ]:
# Normalize using training statistics only
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
train_norm, val_norm, test_norm, norm_params = normalize_features(
    splits['train'], splits['val'], splits['test'], exclude_cols
)

feature_cols = [c for c in train_norm.columns if c not in exclude_cols]
print(f"\nUsing {len(feature_cols)} features")

In [ ]:
# Prepare data
X_train = train_norm[feature_cols].values
y_train = train_norm['soh'].values

X_val = val_norm[feature_cols].values  
y_val = val_norm['soh'].values

X_test = test_norm[feature_cols].values
y_test = test_norm['soh'].values

# Combine train+val for sklearn
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.hstack([y_train, y_val])

print(f"Train+Val: {len(X_train_full)}, Test: {len(X_test)}")
print(f"Test SOH range: [{y_test.min():.2f}, {y_test.max():.2f}]")

In [ ]:
# Train baseline models
print("\n" + "-"*50)
print("Training Baseline Models (Within-Battery Split)")
print("-"*50)

models = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_full, y_train_full)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'mae': mae, 'rmse': rmse, 'r2': r2, 'preds': y_pred}
    print(f"\n{name}:")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['preds'], alpha=0.6, s=40)
    
    min_val, max_val = min(y_test.min(), res['preds'].min()), max(y_test.max(), res['preds'].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    
    ax.set_xlabel('Actual SOH (%)')
    ax.set_ylabel('Predicted SOH (%)')
    ax.set_title(f"{name}\nR² = {res['r2']:.4f}, RMSE = {res['rmse']:.2f}", fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Baseline Models: Within-Battery Time Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/step3_baseline_comparison.png', dpi=150)
plt.show()

In [ ]:
# Summary table
print("\n" + "="*70)
print("BASELINE RESULTS: WITHIN-BATTERY SPLIT")
print("="*70)
print(f"{'Model':<20} {'MAE':<10} {'RMSE':<10} {'R²':<10}")
print("-"*50)
for name, res in results.items():
    print(f"{name:<20} {res['mae']:<10.4f} {res['rmse']:<10.4f} {res['r2']:<10.4f}")
print("-"*50)

best_model = max(results.items(), key=lambda x: x[1]['r2'])
print(f"\n✅ Best: {best_model[0]} (R² = {best_model[1]['r2']:.4f})")

# Save results
results_df = pd.DataFrame([
    {'Model': name, 'MAE': r['mae'], 'RMSE': r['rmse'], 'R2': r['r2']}
    for name, r in results.items()
])
results_df.to_csv('results/phase2/step3_baseline_results.csv', index=False)
print("\n✅ Saved: results/phase2/step3_baseline_results.csv")

---
# Step 4: Multimodal Model
---

In [ ]:
from feature_to_text import FeatureToText, convert_features_to_text

print("="*70)
print("STEP 4: MULTIMODAL MODEL")
print("="*70)

# Convert features to text
print("\n1. Feature-to-Text Conversion")
print("-"*40)

# Use original (non-normalized) features for text
text_df = convert_features_to_text(all_features)

# Show examples
for idx in [0, len(text_df)//2, len(text_df)-1]:
    row = text_df.iloc[idx]
    print(f"\nCycle {row['cycle_number']} (SOH: {row['soh']:.1f}%):")
    print(f"  {row['full_description'][:200]}...")

In [ ]:
# Check if sentence-transformers is available
try:
    from sentence_transformers import SentenceTransformer
    HAVE_SBERT = True
    print("\n2. LLM Embeddings (Sentence-BERT)")
    print("-"*40)
except ImportError:
    HAVE_SBERT = False
    print("\n⚠️ sentence-transformers not installed")
    print("Install with: pip install sentence-transformers")
    print("Using random embeddings for demo...")

In [ ]:
if HAVE_SBERT:
    # Generate real embeddings
    model = SentenceTransformer('all-mpnet-base-v2')
    print(f"Loaded: all-mpnet-base-v2")
    
    texts = text_df['full_description'].tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    
    print(f"\nEmbeddings shape: {embeddings.shape}")
else:
    # Random embeddings for demo
    embeddings = np.random.randn(len(text_df), 768).astype(np.float32)
    print(f"Random embeddings shape: {embeddings.shape}")

In [ ]:
# Add embeddings to features and train a model
print("\n3. Multimodal Training")
print("-"*40)

# Create combined features (numeric + semantic)
# For simplicity, we'll use PCA on embeddings to reduce dimension
from sklearn.decomposition import PCA

pca = PCA(n_components=50)
semantic_features = pca.fit_transform(embeddings)
print(f"Semantic features (PCA): {semantic_features.shape}")

# Combine with numeric features
X_numeric = all_features[feature_cols].values
X_combined = np.hstack([X_numeric, semantic_features])
y_all = all_features['soh'].values

print(f"Combined features: {X_combined.shape}")

In [ ]:
# Split using same indices as before
from sklearn.model_selection import train_test_split

# Simple 70-15-15 split
n = len(X_combined)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

# Sort by battery and cycle to maintain temporal order
sort_idx = np.lexsort((all_features['cycle_number'].values, all_features['battery_id'].values))

X_sorted = X_combined[sort_idx]
y_sorted = y_all[sort_idx]
X_numeric_sorted = X_numeric[sort_idx]

# Per-battery time split
train_idx, val_idx, test_idx = [], [], []

for bat in all_features['battery_id'].unique():
    bat_mask = all_features['battery_id'].values[sort_idx] == bat
    bat_indices = np.where(bat_mask)[0]
    
    n_bat = len(bat_indices)
    train_end_bat = int(n_bat * 0.7)
    val_end_bat = int(n_bat * 0.85)
    
    train_idx.extend(bat_indices[:train_end_bat])
    val_idx.extend(bat_indices[train_end_bat:val_end_bat])
    test_idx.extend(bat_indices[val_end_bat:])

# Create datasets
X_train_mm = X_sorted[train_idx]
y_train_mm = y_sorted[train_idx]
X_val_mm = X_sorted[val_idx]
y_val_mm = y_sorted[val_idx]
X_test_mm = X_sorted[test_idx]
y_test_mm = y_sorted[test_idx]

# Numeric only
X_train_num = X_numeric_sorted[train_idx]
X_test_num = X_numeric_sorted[test_idx]

print(f"Train: {len(X_train_mm)}, Val: {len(X_val_mm)}, Test: {len(X_test_mm)}")

In [ ]:
# Train multimodal vs numeric-only
print("\nTraining comparison...")

# Numeric-only (baseline)
X_train_num_full = np.vstack([X_train_num, X_numeric_sorted[val_idx]])
y_train_num_full = np.hstack([y_train_mm, y_val_mm])

gb_numeric = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gb_numeric.fit(X_train_num_full, y_train_num_full)
y_pred_numeric = gb_numeric.predict(X_test_num)

# Multimodal (numeric + semantic)
X_train_mm_full = np.vstack([X_train_mm, X_sorted[val_idx]])

gb_multimodal = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gb_multimodal.fit(X_train_mm_full, y_train_num_full)
y_pred_multimodal = gb_multimodal.predict(X_test_mm)

# Results
print("\n" + "="*60)
print("MULTIMODAL vs NUMERIC-ONLY COMPARISON")
print("="*60)

print(f"\nNumeric-Only:")
print(f"  MAE:  {mean_absolute_error(y_test_mm, y_pred_numeric):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_mm, y_pred_numeric)):.4f}")
print(f"  R²:   {r2_score(y_test_mm, y_pred_numeric):.4f}")

print(f"\nMultimodal (Numeric + Semantic):")
print(f"  MAE:  {mean_absolute_error(y_test_mm, y_pred_multimodal):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_mm, y_pred_multimodal)):.4f}")
print(f"  R²:   {r2_score(y_test_mm, y_pred_multimodal):.4f}")

# Improvement
rmse_num = np.sqrt(mean_squared_error(y_test_mm, y_pred_numeric))
rmse_mm = np.sqrt(mean_squared_error(y_test_mm, y_pred_multimodal))
improvement = (rmse_num - rmse_mm) / rmse_num * 100

print(f"\nImprovement: {improvement:.2f}%")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Numeric only
ax = axes[0]
ax.scatter(y_test_mm, y_pred_numeric, alpha=0.6, s=40)
min_val, max_val = min(y_test_mm.min(), y_pred_numeric.min()), max(y_test_mm.max(), y_pred_numeric.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
ax.set_xlabel('Actual SOH (%)')
ax.set_ylabel('Predicted SOH (%)')
ax.set_title(f"Numeric-Only\nR² = {r2_score(y_test_mm, y_pred_numeric):.4f}", fontweight='bold')
ax.grid(True, alpha=0.3)

# Multimodal
ax = axes[1]
ax.scatter(y_test_mm, y_pred_multimodal, alpha=0.6, s=40, color='green')
min_val, max_val = min(y_test_mm.min(), y_pred_multimodal.min()), max(y_test_mm.max(), y_pred_multimodal.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
ax.set_xlabel('Actual SOH (%)')
ax.set_ylabel('Predicted SOH (%)')
ax.set_title(f"Multimodal (Numeric + Semantic)\nR² = {r2_score(y_test_mm, y_pred_multimodal):.4f}", fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Step 4: Multimodal Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/step4_multimodal_comparison.png', dpi=150)
plt.show()

---
# Final Summary
---

In [ ]:
print("\n" + "="*70)
print("PHASE-2 STEPS 1-4 COMPLETE")
print("="*70)

print("\n✅ Step 1: Data Audit")
for bat_id, result in audit_results.items():
    status = "PASSED" if result['passed'] else "FAILED"
    print(f"   {bat_id}: {status}")

print(f"\n✅ Step 2: Harmonization")
print(f"   Total samples: {len(features_df)}")
print(f"   Features: {len(feature_cols)}")

print(f"\n✅ Step 3: Baselines (Within-Battery Split)")
for name, res in results.items():
    print(f"   {name}: R² = {res['r2']:.4f}")

print(f"\n✅ Step 4: Multimodal")
print(f"   Numeric-Only R²: {r2_score(y_test_mm, y_pred_numeric):.4f}")
print(f"   Multimodal R²:   {r2_score(y_test_mm, y_pred_multimodal):.4f}")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("5. Ablation Studies")
print("6. Cross-Dataset Generalization")
print("7. Robustness Testing")
print("8. Explainability")
print("9. Statistical Checks")
print("10. Final Report")